In [ ]:
import papermill as pm
from jaxcmr.helpers import find_project_root
from IPython.display import Image, display
import os

# Categorized Serial Position Curve
Plot SPC curves factored by categorical features (e.g., emotion tags).

In [ ]:
base_params = {
    "data_name": "TalmiEEG",
    "data_query": "data['subject'] > 0",
    "category_field": "condition",
    "contrast_name": "condition",
    "category_values": [1, 2],
    "labels": ["Negative", "Neutral"],
    "color_cycle": ["red", "black"],
    "ylim": [0, .8],
    "output_path": "projects/TalmiEEG/results/figures/reference_catspc.png",
}

pm.execute_notebook(
    f"{find_project_root()}/templates/cat_spc.ipynb",
    f"{find_project_root()}/projects/TalmiEEG/notebooks/cat_spc.ipynb",
    parameters=base_params,
)

print(f"![]({base_params['output_path']})")
display(Image(filename=base_params["output_path"]))

# Categorized LPP by Study Position

Plot LPP amplitude by study position, factored by categorical features (e.g., emotion tags).

In [ ]:
base_params = {
    "data_name": "TalmiEEG",
    "data_query": "data['subject'] > 0",
    "category_field": "condition",
    "category_values": [1, 2],
    "labels": ["Negative", "Neutral"],
    "color_cycle": ["red", "black"],
    "ylim": [0, 1.5],
    "output_path": "projects/TalmiEEG/results/figures/cat_lpp_spc_{}.png",
}

for lpp_field in ["EarlyLPP", "LateLPP"]:
    pm.execute_notebook(
        f"{find_project_root()}/templates/cat_lpp_spc.ipynb",
        f"{find_project_root()}/projects/TalmiEEG/notebooks/cat_lpp_spc_{lpp_field}.ipynb",
        parameters=base_params
        | {
            "lpp_field": lpp_field,
            "output_path": base_params["output_path"].format(lpp_field),
        },
    )
    print(f"![]({base_params['output_path'].format(lpp_field)})")
    display(Image(filename=base_params["output_path"].format(lpp_field)))

# Categorized Recall Rate by LPP

Plot the recall rate as a function of binned LPP amplitude for different categories.


In [ ]:
base_params = {
    "data_name": "TalmiEEG",
    "data_query": "data['subject'] > 0",
    "category_field": "condition",
    "category_values": [1, 2],
    "labels": ["Negative", "Neutral"],
    "color_cycle": ["red", "black"],
    "ylim": [0, 1.0],
    "output_path": "projects/TalmiEEG/results/figures/cat_recall_by_lpp_{}.png",
}

for lpp_field in ["EarlyLPP", "LateLPP"]:
    output_path = base_params["output_path"].format(lpp_field)
    pm.execute_notebook(
        f"{find_project_root()}/templates/cat_recall_by_lpp.ipynb",
        f"{find_project_root()}/projects/TalmiEEG/notebooks/cat_recall_by_lpp_{lpp_field}.ipynb",
        parameters=base_params
        | {
            "lpp_field": lpp_field,
            "output_path": output_path,
        },
    )
    print(f"![]({output_path})")
    display(Image(filename=output_path))

    print(f"![]({output_path.replace('.png', '_counts.png')})")
    display(Image(filename=output_path.replace(".png", "_counts.png")))

# Categorized LPP by Recall Status
Plot the LPP amplitudes for items of a given category, separated by whether they were later recalled or not.

In [ ]:
base_params = {
    "data_name": "TalmiEEG",
    "data_query": "data['subject'] > 0",
    "category_field": "condition",
    "labels": ["Recalled", "Unrecalled"],
    "ylim": [-.6, 2.2],
    "output_path": "projects/TalmiEEG/results/figures/cat_lpp_by_recall_{}_{}.png",
}

for category_value, contrast_name in zip([1, 2], ["Negative", "Neutral"]):
    for lpp_field in ["EarlyLPP", "LateLPP"]:
        output_path = base_params["output_path"].format(lpp_field, contrast_name)
        pm.execute_notebook(
            f"{find_project_root()}/templates/cat_lpp_by_recall.ipynb",
            f"{find_project_root()}/projects/TalmiEEG/notebooks/cat_lpp_by_recall_{lpp_field}_{contrast_name}.ipynb",
            parameters=base_params
            | {
                "category_value": category_value,
                "contrast_name": contrast_name,
                "lpp_field": lpp_field,
                "output_path": output_path,
            },
        )
        print(f"![]({output_path})")
        display(Image(filename=output_path))

# Parameter Sensitivity
To confirm expectations about how model parameters affect behavior, we can plot simulations of behavioral benchmarks  across a range of parameter values.

In [ ]:
base_params = {
    # Flow toggles
    "redo_fits": False,
    "redo_sims": False,
    "redo_figures": False,
    "handle_elis": False,
    "filter_repeated_recalls": True,
    # Run configuration
    "base_run_tag": "cat_spc_mse_fixed_term",
    "experiment_count": 200,
    "max_subjects": 0,
    # Data parameters
    "base_data_tag": "TalmiEEG",
    "data_tag": "TalmiEEG",
    "data_path": "data/TalmiEEG.h5",
    "trial_query": "data['subject'] > -1",
    "target_directory": "projects/TalmiEEG/results/",
    # algorithm selection
    "component_paths": {
        "mfc_create_fn": "jaxcmr.components.linear_memory.init_mfc",
        "mcf_create_fn": "jaxcmr.components.linear_memory.init_mcf",
        "context_create_fn": "jaxcmr.components.context.init",
        "termination_policy_create_fn": "jaxcmr.components.termination.NoStopTermination",
    },
    "sim_alg_path": "jaxcmr.simulation.simulate_study_free_recall_and_forced_stop",
    "loss_fn_path": "jaxcmr.cat_spc_mse_loss.MemorySearchCatSpcMseFnGenerator",
    "fit_alg_path": "jaxcmr.fitting.ScipyDE",
    # hyperparameters
    "seed": 0,
    "relative_tolerance": 0.001,
    "popsize": 15,
    "num_steps": 1000,
    "cross_rate": 0.9,
    "diff_w": 0.85,
    "best_of": 3,
    # analysis configuration
    "comparison_analysis_configs": [
        {
            "target": "jaxcmr.analyses.cat_spc.plot_cat_spc",
            "figure_suffix": "cat_spc_negative",
            "kwargs": {"category_field": "condition", "category_values": [1]},
            "ylim": [.3, .75],
        },
        {
            "target": "jaxcmr.analyses.cat_spc.plot_cat_spc",
            "figure_suffix": "cat_spc_neutral",
            "kwargs": {"category_field": "condition", "category_values": [2]},
                        "ylim": [.3, .75],
        },
        {"target": "jaxcmr.analyses.spc.plot_spc", "figure_suffix": "spc", "ylim": [.3, .75]},
                {
            "target": "jaxcmr.analyses.cat_lpp_by_recall.plot_cat_lpp_by_recall",
            "figure_suffix": "cat_lpp_by_recall_NEGATIVE_UNRECALLED",
            "kwargs": {
                "category_field": "condition",
                "category_value": [1],
                "lpp_field": "EarlyLPP",
            },
            "ylim": [-0.6, 2.2],
        },
            {
            "target": "jaxcmr.analyses.cat_lpp_by_recall.plot_cat_lpp_by_recall",
            "figure_suffix": "cat_lpp_by_recall_NEGATIVE_RECALLED",
            "kwargs": {
                "category_field": "condition",
                "category_value": [2],
                "lpp_field": "EarlyLPP",
            },
            "ylim": [-0.6, 2.2],
        },
                    {
            "target": "jaxcmr.analyses.cat_lpp_by_recall.plot_cat_lpp_by_recall",
            "figure_suffix": "cat_lpp_by_recall_NEUTRAL_UNRECALLED",
            "kwargs": {
                "category_field": "condition",
                "category_value": [3],
                "lpp_field": "EarlyLPP",
            },
            "ylim": [-0.6, 2.2],
        },
                            {
            "target": "jaxcmr.analyses.cat_lpp_by_recall.plot_cat_lpp_by_recall",
            "figure_suffix": "cat_lpp_by_recall_NEUTRAL_RECALLED",
            "kwargs": {
                "category_field": "condition",
                "category_value": [4],
                "lpp_field": "EarlyLPP",
            },
            "ylim": [-0.6, 2.2],
        }
    ],
}

In [ ]:
varied_parameters = [
    # Pure-Attention Additive SimpleECMR No-Stop model varying emotion_scale
    {
        "redo_figures": False,
        "redo_fits": False,
        "redo_sims": False,
        "model_name": "AttentionSimpleECMRNoStop",
        "make_factory_path": "jaxcmr.models.attention_simple_ecmr.make_factory",
        "parameters": {
            "fixed": {
                "allow_repeated_recalls": False,
                "learn_after_context_update": False,
                "modulate_emotion_by_primacy": False,
            },
            "free": {
                "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "shared_support": [2.220446049250313e-16, 99.9999999999999998],
                "item_support": [2.220446049250313e-16, 99.9999999999999998],
                "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
                "primacy_scale": [2.220446049250313e-16, 99.9999999999999998],
                "primacy_decay": [2.220446049250313e-16, 99.9999999999999998],
                "choice_sensitivity": [2.220446049250313e-16, 99.9999999999999998],
                "emotion_scale": [2.220446049250313e-16, 9.9999999999999998],
            },
        },
        "varied_parameter": "emotion_scale",
        "sweep_min": 0.0,
        "sweep_max": 10.0,
    },
        # Pure-Attention Multiplicative SimpleECMR No-Stop model varying emotion_scale
    {
        "redo_figures": False,
        "redo_fits": False,
        "redo_sims": False,
        "model_name": "MultiplicativeAttentionSimpleECMRNoStop",
        "make_factory_path": "jaxcmr.models.attention_simple_ecmr.make_factory",
        "parameters": {
            "fixed": {
                "allow_repeated_recalls": False,
                "learn_after_context_update": False,
                "modulate_emotion_by_primacy": True,
            },
            "free": {
                "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "shared_support": [2.220446049250313e-16, 99.9999999999999998],
                "item_support": [2.220446049250313e-16, 99.9999999999999998],
                "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
                "primacy_scale": [2.220446049250313e-16, 99.9999999999999998],
                "primacy_decay": [2.220446049250313e-16, 99.9999999999999998],
                "choice_sensitivity": [2.220446049250313e-16, 99.9999999999999998],
                "emotion_scale": [2.220446049250313e-16, 9.9999999999999998],
            },
        },
        "varied_parameter": "emotion_scale",
        "sweep_min": 0.0,
        "sweep_max": 10.0,
    },
    # Dual-Layer Additive SimpleECMR No-Stop model varying emotion_scale
    {
        "redo_figures": False,
        "redo_fits": False,
        "redo_sims": False,
        "model_name": "AdditiveIsolatedArousalSimpleECMRNoStop",
        "make_factory_path": "jaxcmr.models.simple_ecmr.make_factory",
        "parameters": {
            "fixed": {
                "allow_repeated_recalls": False,
                "learn_after_context_update": False,
                "modulate_emotion_by_primacy": False,
            },
            "free": {
                "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "shared_support": [2.220446049250313e-16, 99.9999999999999998],
                "item_support": [2.220446049250313e-16, 99.9999999999999998],
                "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
                "primacy_scale": [2.220446049250313e-16, 99.9999999999999998],
                "primacy_decay": [2.220446049250313e-16, 99.9999999999999998],
                "choice_sensitivity": [2.220446049250313e-16, 99.9999999999999998],
                "emotion_scale": [2.220446049250313e-16, 9.9999999999999998],
            },
        },
        "varied_parameter": "emotion_scale",
        "sweep_min": 0.0,
        "sweep_max": 10.0,
    },
        # Dual-Layer Multiplicative SimpleECMR No-Stop model varying emotion_scale
    {
        "redo_figures": False,
        "redo_fits": False,
        "redo_sims": False,
        "model_name": "MultiplicativeIsolatedArousalSimpleECMRNoStop",
        "make_factory_path": "jaxcmr.models.simple_ecmr.make_factory",
        "parameters": {
            "fixed": {
                "allow_repeated_recalls": False,
                "learn_after_context_update": False,
                "modulate_emotion_by_primacy": True,
            },
            "free": {
                "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "shared_support": [2.220446049250313e-16, 99.9999999999999998],
                "item_support": [2.220446049250313e-16, 99.9999999999999998],
                "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
                "primacy_scale": [2.220446049250313e-16, 99.9999999999999998],
                "primacy_decay": [2.220446049250313e-16, 99.9999999999999998],
                "choice_sensitivity": [2.220446049250313e-16, 99.9999999999999998],
                "emotion_scale": [2.220446049250313e-16, 9.9999999999999998],
            },
        },
        "varied_parameter": "emotion_scale",
        "sweep_min": 0.0,
        "sweep_max": 10.0,
    },
        # Pure-Attention Additive SimpleECMR No-Stop model varying lpp_scale
    {
        "redo_figures": True,
        "redo_fits": False,
        "redo_sims": False,
        "model_name": "EEGAttentionSimpleECMRNoStop",
        "make_factory_path": "jaxcmr.models.eeg_attention_simple_ecmr.make_factory",
        "parameters": {
            "fixed": {
                "allow_repeated_recalls": False,
                "learn_after_context_update": False,
                "modulate_emotion_by_primacy": False,
            },
            "free": {
                "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "shared_support": [2.220446049250313e-16, 99.9999999999999998],
                "item_support": [2.220446049250313e-16, 99.9999999999999998],
                "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
                "primacy_scale": [2.220446049250313e-16, 99.9999999999999998],
                "primacy_decay": [2.220446049250313e-16, 99.9999999999999998],
                "choice_sensitivity": [2.220446049250313e-16, 99.9999999999999998],
                "emotion_scale": [2.220446049250313e-16, 9.9999999999999998],
            },
        },
        "varied_parameter": "lpp_scale",
        "sweep_min": 0.0,
        "sweep_max": 20.0,
    },
            # Pure-Attention Additive SimpleECMR No-Stop model varying lpp_scale
    {
        "redo_figures": True,
        "redo_fits": False,
        "redo_sims": False,
        "model_name": "AdditiveIsolatedArousalSimpleECMRNoStop",
        "make_factory_path": "jaxcmr.models.eeg_simple_ecmr.make_factory",
        "parameters": {
            "fixed": {
                "allow_repeated_recalls": False,
                "learn_after_context_update": False,
                "modulate_emotion_by_primacy": False,
            },
            "free": {
                "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "shared_support": [2.220446049250313e-16, 99.9999999999999998],
                "item_support": [2.220446049250313e-16, 99.9999999999999998],
                "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
                "primacy_scale": [2.220446049250313e-16, 99.9999999999999998],
                "primacy_decay": [2.220446049250313e-16, 99.9999999999999998],
                "choice_sensitivity": [2.220446049250313e-16, 99.9999999999999998],
                "emotion_scale": [2.220446049250313e-16, 9.9999999999999998],
            },
        },
        "varied_parameter": "lpp_scale",
        "sweep_min": 0.0,
        "sweep_max": 20.0,
    },
]

In [ ]:
for partial_params in varied_parameters:
    params = base_params.copy() | partial_params
    data_tag = params["data_tag"]
    model_name = params["model_name"]
    varied_parameter = partial_params["varied_parameter"]
    base_run_tag = params["base_run_tag"]
    best_of = params["best_of"]
    run_tag = f"{base_run_tag}_best_of_{best_of}"
    max_subjects = params["max_subjects"]
    if max_subjects:
        run_tag += f"_nsubs_{max_subjects}"

    output_path = f"{find_project_root()}/projects/TalmiEEG/notebooks/parameter_shifting_{data_tag}_{model_name}_{run_tag}_shift_{varied_parameter}.ipynb"
    print(output_path)

    pm.execute_notebook(
        f"{find_project_root()}/templates/parameter_shifting.ipynb",
        output_path,
        parameters=params,
        progress_bar=True,
    )

    for analysis_cfg in params["comparison_analysis_configs"]:
        figure_str = f"{data_tag}_{model_name}_{run_tag}_{analysis_cfg['figure_suffix']}_shift_{varied_parameter}.png"
        figure_path = os.path.join(
            f"{find_project_root()}/projects/TalmiEEG/results/figures/{figure_str}"
        )
        print(f"![]({figure_path})")
        display(Image(filename=figure_path))

# Model Fitting
Find the best fitting parameters for a model under specified constraints, simulate the model from the fitted parameters and specified trial scheme, and generate benchmark diagnostics. 

In [ ]:
base_params = {
    # Flow toggles
    "redo_fits": False,
    "redo_sims": False,
    "redo_figures": False,
    "handle_elis": False,
    "filter_repeated_recalls": True,
    # Run configuration
    "base_run_tag": "cat_spc_mse_fixed_term",
    "experiment_count": 200,
    "max_subjects": 0,
    # Data parameters
    "base_data_tag": "TalmiEEG",
    "data_tag": "TalmiEEG",
    "data_path": "data/TalmiEEG.h5",
    "trial_query": "data['subject'] > -1",
    "target_directory": "projects/TalmiEEG/results/",
    # algorithm selection
    "component_paths": {
        "mfc_create_fn": "jaxcmr.components.linear_memory.init_mfc",
        "mcf_create_fn": "jaxcmr.components.linear_memory.init_mcf",
        "context_create_fn": "jaxcmr.components.context.init",
        "termination_policy_create_fn": "jaxcmr.components.termination.NoStopTermination",
    },
    "sim_alg_path": "jaxcmr.simulation.simulate_study_free_recall_and_forced_stop",
    "loss_fn_path": "jaxcmr.cat_spc_mse_loss.MemorySearchCatSpcMseFnGenerator",
    "fit_alg_path": "jaxcmr.fitting.ScipyDE",
    # hyperparameters
    "seed": 0,
    "relative_tolerance": 0.001,
    "popsize": 15,
    "num_steps": 1000,
    "cross_rate": 0.9,
    "diff_w": 0.85,
    "best_of": 3,
    # analysis configuration
    "comparison_analysis_configs": [
        {
            "target": "jaxcmr.analyses.cat_spc.plot_cat_spc",
            "figure_suffix": "cat_spc_negative",
            "kwargs": {"category_field": "condition", "category_values": [1]},
            "ylim": [.2, .8],
        },
        {
            "target": "jaxcmr.analyses.cat_spc.plot_cat_spc",
            "figure_suffix": "cat_spc_neutral",
            "kwargs": {"category_field": "condition", "category_values": [2]},
            "ylim": [.2, .8],
        },
        {"target": "jaxcmr.analyses.spc.plot_spc", "figure_suffix": "spc"},
        # {"target": "jaxcmr.analyses.crp.plot_crp", "figure_suffix": "crp"},
        # {"target": "jaxcmr.analyses.pnr.plot_pnr", "figure_suffix": "pnr"},
        # {
        #     "target": "jaxcmr.analyses.termination_probability.plot_termination_probability", "figure_suffix": "termination_probability"
        # },
    ],
    "single_analysis_configs": [
        {
            "target": "jaxcmr.analyses.cat_spc.plot_cat_spc",
            "figure_suffix": "cat_spc",
            "kwargs": {
                "category_field": "condition",
                "category_values": [1, 2],
                "labels": ["Negative", "Neutral"],
            },
            "ylim": [.2, .8],
            "color_cycle": ["red", "black"]
        },
        {
            "target": "jaxcmr.analyses.cat_lpp_by_recall.plot_cat_lpp_by_recall",
            "figure_suffix": "cat_lpp_by_recall_NEGATIVE_EARLYLPP",
            "kwargs": {
                "category_field": "condition",
                "labels": ["Recalled", "Unrecalled"],
                "category_value": 1,
                "contrast_name": "Negative",
                "lpp_field": "EarlyLPP",
            },
            "ylim": [-0.6, 2.2],
        },
            {
            "target": "jaxcmr.analyses.cat_lpp_by_recall.plot_cat_lpp_by_recall",
            "figure_suffix": "cat_lpp_by_recall_NEUTRAL_EARLYLPP",
            "kwargs": {
                "category_field": "condition",
                "labels": ["Recalled", "Unrecalled"],
                "category_value": 2,
                "contrast_name": "Neutral",
                "lpp_field": "EarlyLPP",
            },
            "ylim": [-0.6, 2.2],
        }
    ],
}

In [ ]:
varied_parameters = [
    # Pure-Attention SimpleECMR model, Additive Interaction varying emotion_scale
    {
        "redo_figures": False,
        "redo_fits": False,
        "redo_sims": False,
        "model_name": "AttentionSimpleECMRNoStop",
        "make_factory_path": "jaxcmr.models.attention_simple_ecmr.make_factory",
        "parameters": {
            "fixed": {
                "allow_repeated_recalls": False,
                "learn_after_context_update": False,
                "modulate_emotion_by_primacy": False,
            },
            "free": {
                "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "shared_support": [2.220446049250313e-16, 99.9999999999999998],
                "item_support": [2.220446049250313e-16, 99.9999999999999998],
                "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
                "primacy_scale": [2.220446049250313e-16, 99.9999999999999998],
                "primacy_decay": [2.220446049250313e-16, 99.9999999999999998],
                "choice_sensitivity": [2.220446049250313e-16, 99.9999999999999998],
                "emotion_scale": [2.220446049250313e-16, 9.9999999999999998],
            },
        },
    },
        {
        "redo_figures": False,
        "redo_fits": False,
        "redo_sims": False,
        "model_name": "MultiplicativeAttentionSimpleECMRNoStop",
        "make_factory_path": "jaxcmr.models.attention_simple_ecmr.make_factory",
        "parameters": {
            "fixed": {
                "allow_repeated_recalls": False,
                "learn_after_context_update": False,
                "modulate_emotion_by_primacy": True,
            },
            "free": {
                "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "shared_support": [2.220446049250313e-16, 99.9999999999999998],
                "item_support": [2.220446049250313e-16, 99.9999999999999998],
                "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
                "primacy_scale": [2.220446049250313e-16, 99.9999999999999998],
                "primacy_decay": [2.220446049250313e-16, 99.9999999999999998],
                "choice_sensitivity": [2.220446049250313e-16, 99.9999999999999998],
                "emotion_scale": [2.220446049250313e-16, 9.9999999999999998],
            },
        },
    },
            {
        "redo_figures": False,
        "redo_fits": False,
        "redo_sims": False,
        "model_name": "AdditiveIsolatedArousalSimpleECMRNoStop",
        "make_factory_path": "jaxcmr.models.simple_ecmr.make_factory",
        "parameters": {
            "fixed": {
                "allow_repeated_recalls": False,
                "learn_after_context_update": False,
                "modulate_emotion_by_primacy": False,
            },
            "free": {
                "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "shared_support": [2.220446049250313e-16, 99.9999999999999998],
                "item_support": [2.220446049250313e-16, 99.9999999999999998],
                "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
                "primacy_scale": [2.220446049250313e-16, 99.9999999999999998],
                "primacy_decay": [2.220446049250313e-16, 99.9999999999999998],
                "choice_sensitivity": [2.220446049250313e-16, 99.9999999999999998],
                "emotion_scale": [2.220446049250313e-16, 9.9999999999999998],
            },
        },
    },
                {
        "redo_figures": False,
        "redo_fits": False,
        "redo_sims": False,
        "model_name": "MultiplicativeIsolatedArousalSimpleECMRNoStop",
        "make_factory_path": "jaxcmr.models.simple_ecmr.make_factory",
        "parameters": {
            "fixed": {
                "allow_repeated_recalls": False,
                "learn_after_context_update": False,
                "modulate_emotion_by_primacy": True,
            },
            "free": {
                "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "shared_support": [2.220446049250313e-16, 99.9999999999999998],
                "item_support": [2.220446049250313e-16, 99.9999999999999998],
                "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
                "primacy_scale": [2.220446049250313e-16, 99.9999999999999998],
                "primacy_decay": [2.220446049250313e-16, 99.9999999999999998],
                "choice_sensitivity": [2.220446049250313e-16, 99.9999999999999998],
                "emotion_scale": [2.220446049250313e-16, 9.9999999999999998],
            },
        },
    },
                    {
        "redo_figures": False,
        "redo_fits": False,
        "redo_sims": False,
        "model_name": "SumInteractionSimpleECMRNoStop",
        "make_factory_path": "jaxcmr.models.simple_ecmr.make_factory",
        "parameters": {
            "fixed": {
                "allow_repeated_recalls": False,
                "learn_after_context_update": False,
                "modulate_emotion_by_primacy": True,
            },
            "free": {
                "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "shared_support": [2.220446049250313e-16, 99.9999999999999998],
                "item_support": [2.220446049250313e-16, 99.9999999999999998],
                "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
                "primacy_scale": [2.220446049250313e-16, 99.9999999999999998],
                "primacy_decay": [2.220446049250313e-16, 99.9999999999999998],
                "choice_sensitivity": [2.220446049250313e-16, 99.9999999999999998],
                "emotion_scale": [2.220446049250313e-16, 9.9999999999999998],
            },
        },
    },
        {
        "redo_figures": True,
        "redo_fits": True,
        "redo_sims": True,
        "model_name": "EEGAttentionSimpleECMRNoStop",
        "make_factory_path": "jaxcmr.models.eeg_attention_simple_ecmr.make_factory",
        "parameters": {
            "fixed": {
                "allow_repeated_recalls": False,
                "learn_after_context_update": False,
                "modulate_emotion_by_primacy": False,
            },
            "free": {
                "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "shared_support": [2.220446049250313e-16, 99.9999999999999998],
                "item_support": [2.220446049250313e-16, 99.9999999999999998],
                "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
                "primacy_scale": [2.220446049250313e-16, 99.9999999999999998],
                "primacy_decay": [2.220446049250313e-16, 99.9999999999999998],
                "choice_sensitivity": [2.220446049250313e-16, 99.9999999999999998],
                "emotion_scale": [2.220446049250313e-16, 9.9999999999999998],
                "lpp_scale": [2.220446049250313e-16, 19.9999999999999998],
            },
        },
    },
            {
        "redo_figures": True,
        "redo_fits": True,
        "redo_sims": True,
        "model_name": "EEGTwoLayerSimpleECMRNoStop",
        "make_factory_path": "jaxcmr.models.eeg_simple_ecmr.make_factory",
        "parameters": {
            "fixed": {
                "allow_repeated_recalls": False,
                "learn_after_context_update": False,
                "modulate_emotion_by_primacy": False,
            },
            "free": {
                "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "shared_support": [2.220446049250313e-16, 99.9999999999999998],
                "item_support": [2.220446049250313e-16, 99.9999999999999998],
                "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
                "primacy_scale": [2.220446049250313e-16, 99.9999999999999998],
                "primacy_decay": [2.220446049250313e-16, 99.9999999999999998],
                "choice_sensitivity": [2.220446049250313e-16, 99.9999999999999998],
                "emotion_scale": [2.220446049250313e-16, 9.9999999999999998],
                "lpp_scale": [2.220446049250313e-16, 19.9999999999999998],
            },
        },
    },
            {
        "redo_figures": False,
        "redo_fits": False,
        "redo_sims": False,
        "model_name": "WeirdCMRNoStop",
        "make_factory_path": "jaxcmr.models.cmr.make_factory",
        "parameters": {
            "fixed": {
                "allow_repeated_recalls": False,
                "learn_after_context_update": False,
            },
            "free": {
                "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "shared_support": [2.220446049250313e-16, 99.9999999999999998],
                "item_support": [2.220446049250313e-16, 99.9999999999999998],
                "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
                "primacy_scale": [2.220446049250313e-16, 99.9999999999999998],
                "primacy_decay": [2.220446049250313e-16, 99.9999999999999998],
                "choice_sensitivity": [2.220446049250313e-16, 99.9999999999999998],
            },
        },
    },
]

In [ ]:
for partial_params in varied_parameters:
    params = base_params.copy() | partial_params
    data_tag = params["data_tag"]
    model_name = params["model_name"]
    base_run_tag = params["base_run_tag"]
    best_of = params["best_of"]
    run_tag = f"{base_run_tag}_best_of_{best_of}"
    max_subjects = params["max_subjects"]
    if max_subjects:
        run_tag += f"_nsubs_{max_subjects}"

    output_path = f"{find_project_root()}/projects/TalmiEEG/notebooks/fitting_{data_tag}_{model_name}_{run_tag}.ipynb"
    print(output_path)

    pm.execute_notebook(
        f"{find_project_root()}/templates/fitting.ipynb",
        output_path,
        parameters=params,
        progress_bar=True,
    )

    for analysis_cfg in params["single_analysis_configs"]:
        figure_str = f"{data_tag}_{model_name}_{run_tag}_{analysis_cfg['figure_suffix']}.png"
        figure_path = os.path.join(
            f"{find_project_root()}/projects/TalmiEEG/results/figures/{figure_str}"
        )
        print(f"![]({figure_path})")
        display(Image(filename=figure_path))
        
    for analysis_cfg in params["comparison_analysis_configs"]:
        figure_str = f"{data_tag}_{model_name}_{run_tag}_{analysis_cfg['figure_suffix']}.png"
        figure_path = os.path.join(
            f"{find_project_root()}/projects/TalmiEEG/results/figures/{figure_str}"
        )
        print(f"![]({figure_path})")
        display(Image(filename=figure_path))
